# **Bagging Regressor**

In [57]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import BaggingRegressor 
from sklearn.svm import SVR


import warnings
warnings.filterwarnings('ignore')

In [66]:
# importing diabetes dataset
from sklearn import datasets

diabetes = datasets.load_diabetes()
X_diabetes, Y_diabetes = diabetes.data, diabetes.target

print(f"Dataset Features names : {str(diabetes.feature_names)}")
print(f"Dataset Features size : {str(diabetes.data.shape)}")
print(f"Dataset Target size : {str(diabetes.target.shape)}")

diabetes.data

Dataset Features names : ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
Dataset Features size : (442, 10)
Dataset Target size : (442,)


array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
         0.01990749, -0.01764613],
       [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
        -0.06833155, -0.09220405],
       [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
         0.00286131, -0.02593034],
       ...,
       [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
        -0.04688253,  0.01549073],
       [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
         0.04452873, -0.02593034],
       [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
        -0.00422151,  0.00306441]])

In [67]:
X_train, X_test, y_train, y_test = train_test_split(X_diabetes, Y_diabetes, test_size=0.2, random_state=42)

print("\nDATA SET SIZES : ", X_train.shape, X_test.shape, y_train.shape, y_test.shape)

lr = LinearRegression()
dt = DecisionTreeRegressor()
knn = KNeighborsRegressor()


lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
knn.fit(X_train, y_train)

y_pred1 = lr.predict(X_test)
y_pred2 = dt.predict(X_test)
y_pred3 = knn.predict(X_test)


print("LR \t:", np.round(r2_score(y_test, y_pred1), 2))
print("DT \t:", np.round(r2_score(y_test, y_pred2), 2))
print("KNN \t:", np.round(r2_score(y_test, y_pred3), 2))


DATA SET SIZES :  (353, 10) (89, 10) (353,) (89,)
LR 	: 0.45
DT 	: -0.01
KNN 	: 0.43


In [68]:
bag_regressor = BaggingRegressor(
    estimator=DecisionTreeRegressor(),       # even if you comment this line, the would be working perfectly fine. Coz the default estimator for bagging regressor is 'DecisionTreeRegressor'
    n_estimators=500,
    max_samples=0.5,
    bootstrap=True,
    oob_score=True,
    random_state=42
)

bag_regressor.fit(X_train, y_train)
y_preds = bag_regressor.predict(X_test)


print('Training Coefficient of R^2 score : %.3f' %bag_regressor.score(X_train, y_train))
print('Test Coefficient of R^2 score : %.3f' %bag_regressor.score(X_test, y_test))



Training Coefficient of R^2 score : 0.801
Test Coefficient of R^2 score : 0.459


In [ ]:
%%time

n_samples = diabetes.data.shape[0]
n_features = diabetes.data.shape[1]

params = {
    'estimator' : [None, LinearRegression(), KNeighborsRegressor()],
    'n_estimators' : [20,50,100],
    'max_samples': [0.5,1.0],
    'max_features': [0.5,1.0],
    'bootstrap': [True, False],
    'bootstrap_features': [True, False]
}



bagging_regressor_grid = GridSearchCV(
    BaggingRegressor(
        estimator=DecisionTreeRegressor(),       # even if you comment this line, the would be working perfectly fine. Coz the default estimator for bagging regressor is 'DecisionTreeRegressor'
        n_estimators=50,
        max_samples=0.5,
        bootstrap=True,
        oob_score=True,
        random_state=42), 
    param_grid=params, 
    cv=3, 
    n_jobs=-1, 
    verbose=1)

# so in params we have put all the values which will be used inside BaggingRegressor as a parameter, so what the model will do is, it will take all the model one by one and check which one gives teh best resualts

bagging_regressor_grid.fit(X_train, y_train)

print('Train R^2 Score : %.3f'%bagging_regressor_grid.best_estimator_.score(X_train, y_train))
print('Test R^2 Score : %.3f'%bagging_regressor_grid.best_estimator_.score(X_test, y_test))
print('Best R^2 Score Through Grid Search : %.3f'%bagging_regressor_grid.best_score_)
print('Best Parameters : ',bagging_regressor_grid.best_params_)





Fitting 3 folds for each of 144 candidates, totalling 432 fits
Train R^2 Score : 0.527
Test R^2 Score : 0.445
Best R^2 Score Through Grid Search : 0.490
Best Parameters :  {'bootstrap': True, 'bootstrap_features': False, 'estimator': LinearRegression(), 'max_features': 1.0, 'max_samples': 0.5, 'n_estimators': 50}
CPU times: total: 625 ms
Wall time: 16.7 s


```python
params = [
    # 1. Tuning for the default Decision Tree (None)
    {
        'estimator': [None],
        'n_estimators': [20, 50, 100],
        'max_samples': [0.5, 1.0],
        'bootstrap': [True, False]
    },
    
    # 2. Tuning for KNN (using estimator__ to reach inside)
    {
        'estimator': [KNeighborsRegressor()],
        'estimator__n_neighbors': [3, 5, 7, 9],  # Tuning the KNN brain
        'estimator__weights': ['uniform', 'distance'],
        'n_estimators': [20, 50],
        'max_samples': [0.5, 1.0],
        'bootstrap': [True]
    },
    
    # 3. Tuning for Linear Regression
    {
        'estimator': [LinearRegression()],
        'estimator__fit_intercept': [True, False], # Tuning the LR brain
        'n_estimators': [20, 50],
        'max_samples': [0.5, 1.0],
        'bootstrap': [True]
    }
]
# Initialize the grid search
# We pass a base BaggingRegressor() as the primary model
bagging_regressor_grid = GridSearchCV(
    BaggingRegressor(random_state=42), 
    param_grid=params, 
    cv=3,              # 3-Fold Cross Validation
    scoring='r2',      # We want to maximize the R-squared score
    n_jobs=-1,         # Use all CPU cores for speed
    verbose=1          # Show progress updates
)
```

### **What is param_grid ???**

When we train a model, we often don't know the best settings (hyperparameters). Should we use 50 trees or 100? Should we use linear regression or a decision tree? Instead of guessing, we list all the possibilities in a dictionary.

`GridSearchCV` takes your `params` dictionary and calculates every single possible combination of those values.

What we have used above for `param_grid`: 
- 3 estimators
- 3 n_estimators counts
- 2 max_samples values
- 2 max_features values
- 2 bootstrap options
- 2 bootstrap_features options




$3 \times 3 \times 2 \times 2 \times 2 \times 2 = \mathbf{144 \text{ different models!}}$

### **Detailed breakdown of `params`**

In the `params` dictionary, we specify the various hyperparameters we want to tune. Each key represents a parameter, and the list of values represents the different options we want the robot to test.

**What individual parameters mean here:**
- **`estimator`**: Tests different "brains" (Decision Trees, Linear Regression, or KNN) to see which algorithm performs best as part of the ensemble.
- **`n_estimators`**: Tests how many individual models (20, 50, or 100) are needed to make the prediction stable and accurate.
- **`max_samples`**: Controls row diversity. `0.5` means each tree only sees 50% of the rows, while `1.0` means they see all of them.
- **`max_features`**: Controls column diversity. It tests if the model performs better by only looking at a subset (50%) of the features/columns.
- **`bootstrap`**: If `True`, it uses sampling with replacement (Bagging). If `False`, it uses sampling without replacement (Pasting).
- **`bootstrap_features`**: Tests if we should sample the columns/features with replacement as well.

`GridSearchCV` will test every single combination of these settings and find the one that results in the highest R² score.

**So bascially here i am using 4 models, decision tree, none, linear reg, and knn ???**

In our params dictionary, we are testing 3 different algorithms (brains) to see which one works best for this specific dataset.

Note on `None`: In Scikit-Learn, setting estimator to None simply tells the model to use its built-in default, which is a Decision Tree.

The 3 models being tested are:

- Decision Tree: (Represented by `None`)
- Linear Regression: (Represented by `LinearRegression()`)
- KNN: (Represented by `KNeighborsRegressor()`)

--- 

The load_diabetes dataset is notoriously difficult. Unlike the "Iris" dataset which is easy to get 95%+ accuracy on, the Diabetes dataset is a real-world regression problem where the features (age, BMI, blood pressure) only partially explain the progression of the disease.

Standard Performance: Usually, a solid Linear Regression on this dataset gets an R² between 0.40 and 0.50.
Your Result: Getting 0.46 or 0.49 means your model is actually performing "normally" for this specific data.
